# Python Collections 

| Priority | Tool | What it gives you | Major interview patterns |
|---|---|---|---|
| ⭐⭐⭐⭐⭐ | `defaultdict` | Dictionary with automatic defaults | Grouping, adjacency lists, counting, aggregation |
| ⭐⭐⭐⭐⭐ | `Counter` | Frequency map | Frequency counting, top-K, anagrams, sliding window |
| ⭐⭐⭐⭐⭐ | `deque` | O(1) operations at both ends | BFS, queues, sliding window, monotonic queue |
| ⭐⭐⭐⭐ | `OrderedDict` | Explicit ordered-map operations | LRU cache, ordering |
| ⭐⭐⭐ | `namedtuple` | Lightweight immutable records | Structured data, coordinates/states |
| ⭐⭐ | `ChainMap` | Layer multiple dictionaries | Config/lookup hierarchy |

### OrderedDict 

OrderedDict was primarily used because the standard python dictionary did not preserve the insertion order.

#### Most popular problem used in here is the LRU 


Differences between orderedDict and dict Objects 

The regular dict was designed to be very good at mapping operations. Tracking insertion order was secondary.

The OrderedDict was designed to be good at reordering operations. Space efficiency, iteration speed, and the performance of update operations were secondary.

The OrderedDict algorithm can handle frequent reordering operations better than dict. As shown in the recipes below, this makes it suitable for implementing various kinds of LRU caches.

The equality operation for OrderedDict checks for matching order.

A regular dict can emulate the order sensitive equality test with p == q and all(k1 == k2 for k1, k2 in zip(p, q)).

The popitem() method of OrderedDict has a different signature. It accepts an optional argument to specify which item is popped.

A regular dict can emulate OrderedDict’s od.popitem(last=True) with d.popitem() which is guaranteed to pop the rightmost (last) item.

A regular dict can emulate OrderedDict’s od.popitem(last=False) with (k := next(iter(d)), d.pop(k)) which will return and remove the leftmost (first) item if it exists.

OrderedDict has a move_to_end() method to efficiently reposition an element to an endpoint.

A regular dict can emulate OrderedDict’s od.move_to_end(k, last=True) with d[k] = d.pop(k) which will move the key and its associated value to the rightmost (last) position.

A regular dict does not have an efficient equivalent for OrderedDict’s od.move_to_end(k, last=False) which moves the key and its associated value to the leftmost (first) position.



Methods supported by OrderedDict 

- popitem(last=True)
    The popitem() method for ordered dictionaries returns and removes a (key, value) pair. The pairs are returned in LIFO order if last is true or FIFO order if false.

- move_to_end(key, last=True) 
    Move an existing key to either end of an ordered dictionary. The item is moved to the right end if last is true (the default) or to the beginning if last is false


OrderedDict Examples and Recipes 

In [3]:
from collections import OrderedDict

d = OrderedDict.fromkeys('xbcdea')
print (d)

d.move_to_end('x')
print (d)

d.move_to_end('a', last=False)
print (d)

OrderedDict({'x': None, 'b': None, 'c': None, 'd': None, 'e': None, 'a': None})
OrderedDict({'b': None, 'c': None, 'd': None, 'e': None, 'a': None, 'x': None})
OrderedDict({'a': None, 'b': None, 'c': None, 'd': None, 'e': None, 'x': None})


In [ ]:
""" 
It is straightforward to create an ordered dictionary variant that remembers the order the keys were last 
inserted. If a new entry overwrites an existing entry, the original insertion position is changed and moved 
to the end:
"""
class LastUpdatedOrderedDict(OrderedDict):
    'store items in the order the keys were last added'

    def __setitem__(self, key, value):
        super().__setitem__(key, value)
        self.move_to_end(key)
    

### LRU implementations using OrderedDict

In [ ]:
from collections import OrderedDict
from time import monotonic

class TimeBoundedLRU:
    "LRU Cache that invalidates and refreshes old entries."

    def __init__(self, func, maxsize=128, maxage=30):
        self.cache = OrderedDict()      # { args : (timestamp, result)}
        self.func = func
        self.maxsize = maxsize
        self.maxage = maxage

    def __call__(self, *args):
        if args in self.cache:
            self.cache.move_to_end(args)
            timestamp, result = self.cache[args]
            if monotonic() - timestamp <= self.maxage:
                return result
        result = self.func(*args)
        self.cache[args] = monotonic(), result
        if len(self.cache) > self.maxsize:
            self.cache.popitem(last=False)
        return result
    

In [ ]:
class MultiHitLRUCache:
    """ LRU cache that defers caching a result until
        it has been requested multiple times.

        To avoid flushing the LRU cache with one-time requests,
        we don't cache until a request has been made more than once.

    """

    def __init__(self, func, maxsize=128, maxrequests=4096, cache_after=1):
        self.requests = OrderedDict()   # { uncached_key : request_count }
        self.cache = OrderedDict()      # { cached_key : function_result }
        self.func = func
        self.maxrequests = maxrequests  # max number of uncached requests
        self.maxsize = maxsize          # max number of stored return values
        self.cache_after = cache_after

    def __call__(self, *args):
        if args in self.cache:
            self.cache.move_to_end(args)
            return self.cache[args]
        result = self.func(*args)
        self.requests[args] = self.requests.get(args, 0) + 1
        if self.requests[args] <= self.cache_after:
            self.requests.move_to_end(args)
            if len(self.requests) > self.maxrequests:
                self.requests.popitem(last=False)
        else:
            self.requests.pop(args, None)
            self.cache[args] = result
            if len(self.cache) > self.maxsize:
                self.cache.popitem(last=False)
        return result
    